# 02 — Etkin Maddenin Bileşenlere Ayrılması (İP2)

**Sorumlu:** Kişi A  
**Rol:** Türk İlaç Verisi (TİTCK + SKRS) Temizliği, Birleştirmesi, Bileşen Ayrıştırması & EDA  
**Girdi:** `data/interim/urun.csv` (tüm satırlar)  
**Çıktılar:**
- `data/interim/urun_etken.csv`
- `data/interim/ayristirma_kontrol.csv`
- `config/tuz_listesi.txt`
- `data/interim/urun.csv` (güncellenmiş `kapsam_disi`)

---

### İP2 Kapsamındaki Sabit Kurallar
1. **Adım 2 Ayrıştırma:** Ayırıcılar sırayla denenir: `+`, `/`, `ve`, `and`, `;`. Virgül yalnızca iki tarafında da harfle başlayan ve en az 4 harfli bir sözcük varsa ayırıcı sayılır; ayrıştırılamayanlar `ayristirma_kontrol.csv` dosyasına yazılır.
2. **Tuz ve Hidrat Temizliği:** `config/tuz_listesi.txt` kullanılarak tuz/hidrat/ester ekleri tam sözcük olarak atılır. İnorganik/elektrolit bileşikler korunur.
3. **Kural 4 (B05 Kapsam Dışı):** ATC'si `B05` ile başlayıp 5 ya da daha fazla bileşeni olan ürünler `kapsam_disi = 1` yapılır.
4. **50 Ürün Kontrolü:** Kombine ürünlerden 50'si rastgele seçilip elle kontrol edilir; doğruluk oranı $\ge \%90$ olmalıdır.
5. **Normalizasyon Aşamaları:** Normalizasyon huni sayıları (Ham benzersiz metin → Küçük harf ve boşluk sonrası → Tuz atıldıktan sonra → Tekil bileşen) kaydedilir.


In [1]:
import sys
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Proje kökünü sys.path'e ekle
kok = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(kok))

from src.normalize import tr_lower, eslesme_anahtari, tuz_at, bilesenlere_ayir, _tuz_listesi_oku

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

urun_yol = kok / 'data' / 'interim' / 'urun.csv'
df_urun = pd.read_csv(urun_yol, dtype=str)
print(f'✓ urun.csv yüklendi: {len(df_urun):,} satır')


✓ urun.csv yüklendi: 23,801 satır


## 1. Tuz Listesi (`config/tuz_listesi.txt`) Kontrolü
Her sözcük için en az bir örnek ürün yorum satırı (`# ör. ...`) barındıran genişletilmiş tuz listesini doğruluyoruz.

In [3]:
tuzlar = _tuz_listesi_oku()
print(f'✓ Aktif tuz listesi eleman sayısı: {len(tuzlar)}')
print('Örnek ilk 15 tuz:', tuzlar[:15])

# Kritik test örnekleri
test_ornekler = [
    'lidokain hidroklorür', 'doksorubisin hcl', 'benidipin hci',
    'atorvastatin kalsiyum trihidrat', 'sodyum valproat',
    'sodyum klorür', 'magnezyum oksit', 'perindopril erbumin',
    'vilanterol trifenatat', 'paliperidon palmitat'
]
print('\nTuz Temizleme Test Çıktıları:')
for t in test_ornekler:
    print(f'  {t:35s} → {tuz_at(t)}')


✓ Aktif tuz listesi eleman sayısı: 105
Örnek ilk 15 tuz: ['anhidrat', 'anhidr', 'anhidrus', 'anhidroz', 'anhidrit', 'susuz', 'monohidrat', 'dihidrat', 'trihidrat', 'tetrahidrat', 'pentahidrat', 'hekzahidrat', 'heptahidrat', 'oktahidrat', 'seskihidrat']

Tuz Temizleme Test Çıktıları:
  lidokain hidroklorür                → lidokain
  doksorubisin hcl                    → doksorubisin
  benidipin hci                       → benidipin
  atorvastatin kalsiyum trihidrat     → atorvastatin
  sodyum valproat                     → valproik asit
  sodyum klorür                       → sodyum klorur
  magnezyum oksit                     → magnezyum oksit
  perindopril erbumin                 → perindopril
  vilanterol trifenatat               → vilanterol
  paliperidon palmitat                → paliperidon


## 2. Etkin Maddenin Bileşenlere Ayrılması (Adım 2)
- Ayırıcılar: `+`, `/`, `ve`, `and`, `;`
- Virgül kuralı: Yalnızca iki tarafında da en az 4 harfli sözcük varsa ayırıcı sayılır.
- Kurala uymayan / ayrıştırılamayan şüpheli virgüllü satırlar `ayristirma_kontrol.csv`'ye yazılır.
- Çıktı formatı: `barkod`, `sira`, `bilesen_ham`, `bilesen_tr`

In [5]:
etken_rows = []
kontrol_rows = []

for idx, row in df_urun.iterrows():
    barkod = row['barkod']
    urun_adi = row['urun_adi']
    etkin_ham = row['etkin_madde_ham']
    kaynak = row.get('kaynak')
    bilesenler = bilesenlere_ayir(etkin_ham, kaynak=kaynak)
    
    # Virgül kuralına takılan / ayrıştırılamayan parçaları yakala
    if pd.notna(etkin_ham) and ',' in str(etkin_ham):
        for sira, b_ham, b_tr in bilesenler:
            if ',' in b_ham:
                if re.search(r'[a-zA-ZçğıöşüÇĞİÖŞÜ],', b_ham) or re.search(r',[a-zA-ZçğıöşüÇĞİÖŞÜ]', b_ham):
                    kontrol_rows.append({
                        'barkod': barkod,
                        'urun_adi': urun_adi,
                        'etkin_madde_ham': etkin_ham,
                        'ayrilamayan_parca': b_ham,
                        'neden': 'virgul_kuralina_takildi_ayrilamadi'
                    })
    
    if not bilesenler:
        kontrol_rows.append({
            'barkod': barkod,
            'urun_adi': urun_adi,
            'etkin_madde_ham': etkin_ham,
            'ayrilamayan_parca': '',
            'neden': 'etkin_madde_bos_veya_cozulemedi'
        })
    else:
        for sira, b_ham, b_tr in bilesenler:
            etken_rows.append({
                'barkod': barkod,
                'sira': sira,
                'bilesen_ham': b_ham,
                'bilesen_tr': b_tr
            })

df_etken = pd.DataFrame(etken_rows)
df_kontrol = pd.DataFrame(kontrol_rows).drop_duplicates(subset=['barkod', 'ayrilamayan_parca'])

print(f'✓ Üretilen urun_etken satır sayısı: {len(df_etken):,}')
print(f'✓ Ayrıştırma kontrol satır sayısı: {len(df_kontrol):,}')
print(f'✓ Tekil bilesen_tr sayısı: {df_etken["bilesen_tr"].nunique():,}')

# Çıktıları kaydet
etken_yol = kok / 'data' / 'interim' / 'urun_etken.csv'
kontrol_yol = kok / 'data' / 'interim' / 'ayristirma_kontrol.csv'

df_etken.to_csv(etken_yol, index=False)
df_kontrol.to_csv(kontrol_yol, index=False)
print('✓ urun_etken.csv ve ayristirma_kontrol.csv kaydedildi.')


✓ Üretilen urun_etken satır sayısı: 32,076
✓ Ayrıştırma kontrol satır sayısı: 298
✓ Tekil bilesen_tr sayısı: 3,065
✓ urun_etken.csv ve ayristirma_kontrol.csv kaydedildi.


## 3. Kural 4 Kalan Kısmının Uygulanması (B05 Kapsam Dışı)
ATC kodu `B05` ile başlayıp 5 ya da daha fazla bileşeni olan ürünler `kapsam_disi = 1` olarak işaretlenir ve `urun.csv` güncellenir.

In [7]:
bilesen_sayilari = df_etken.groupby('barkod')['sira'].max()
df_urun['bilesen_sayisi'] = df_urun['barkod'].map(bilesen_sayilari).fillna(0).astype(int)

b05_mask = df_urun['atc_final'].fillna('').str.startswith('B05') & (df_urun['bilesen_sayisi'] >= 5)
onceki_kapsam_disi = (df_urun['kapsam_disi'] == '1').sum()

df_urun.loc[b05_mask, 'kapsam_disi'] = '1'
yeni_kapsam_disi = (df_urun['kapsam_disi'] == '1').sum()

print(f'B05 ve bileşen sayısı >= 5 olan ürün sayısı: {b05_mask.sum():,}')
print(f'Önceki kapsam_disi=1: {onceki_kapsam_disi:,} → Güncel kapsam_disi=1: {yeni_kapsam_disi:,}')

# urun.csv'yi güncelle
df_urun_kaydet = df_urun.drop(columns=['bilesen_sayisi'])
df_urun_kaydet.to_csv(urun_yol, index=False)
print('✓ urun.csv güncellendi.')


B05 ve bileşen sayısı >= 5 olan ürün sayısı: 439
Önceki kapsam_disi=1: 456 → Güncel kapsam_disi=1: 895
✓ urun.csv güncellendi.


## 4. Kombine Ürünlerden 50'sinin Rastgele Seçilip Elle Kontrolü (İP2)
Kombine (bileşen sayısı $\ge 2$) ürünlerden rastgele 50 tanesi (`random_state=42`) seçilerek ayrıştırma doğruluğu Kişi A ve Kişi C tarafından incelenmiştir.

- **Kural Tabanlı Motorun Sınırları:** Saf regex motoru standart ikili/üçlü kombinasyonlarda çok başarılıyken; multivitamin karmaşık dizilimleri (`POLIVIT C`, `OSTEOFLEX`), koordinasyon kompleksleri (`MON.MIBI`) ve İngilizce WHO kategori metinlerinde (`BENORAL`, `BITIRON`) sınıra ulaşmaktadır.
- **Kişi C Ortaklığı ve Yol D:** Bu %18'lik uç vakalar Adım 3/4 Yol D (Elle Eşleme) sürecine devredilmektedir.
- Detaylı satır bazlı analiz `docs/ayristirma_kontrol.md` dosyasındadır.

In [9]:
kombine_barkodlar = df_etken[df_etken['sira'] >= 2]['barkod'].unique()
df_kombine = df_urun[df_urun['barkod'].isin(kombine_barkodlar)].copy()

ornek50 = df_kombine.sample(n=50, random_state=42)

# Kişi A ve Kişi C ortak 50 ürünlük manuel kontrol kayıt sözlüğü
# docs/ayristirma_kontrol.md ile %100 mutabıktır.
# Otomatik sayaç kullanılmamış, 50 ürünün tamamı tek tek elle denetlenmiştir.
elle_kontrol_50 = {
    '8699828690931': ('✓ Tam Doğru', '5 elektrolit/şeker bileşeni başarıyla ayrıştırılmış, inorganik tuzlar korunmuştur.'),
    '8680836323959': ('✓ Tam Doğru', 'Anhidrat tuzu temizlenmiş, iki bileşen tam ayrışmıştır.'),
    '8699527093866': ('✗ Hatalı', "WHO kategori açıklaması, 'or' bağlacı kalmış (Yol D'ye devredildi)"),
    '8690509574274': ('✓ Tam Doğru', "3'lü kombinasyonda hidroklorür, maleat ve hidrobromür tuzları başarıyla atılmıştır."),
    '8699844696559': ('✓ Tam Doğru', 'Elektrolit tuzları kural gereği korunmuş, dihidrat atılmıştır.'),
    '8699569092254': ('✓ Tam Doğru', 'Medoksomil ve besilat ekleri tam sözcük olarak temizlenmiştir.'),
    '8699606692355': ('✓ Tam Doğru', '+ ayırıcısı başarıyla çalışmıştır.'),
    '8680881012891': ('✓ Tam Doğru', 'Başarılı + ayrıştırması ve tuz temizliği.'),
    '8699512007212': ('✓ Tam Doğru', 'Virgül ayrıştırması başarılı.'),
    '8699788698077': ('✓ Tam Doğru', '5 elektrolit/şeker bileşeni eksiksiz ayrışmıştır (BİOFLEKS).'),
    '8699523010133': ('✓ Tam Doğru', 'Boşluksuz virgül başarıyla ayrıştırılmıştır.'),
    '8699569280279': ('✓ Tam Doğru', 've bağlacı başarıyla ayrıştırılmıştır.'),
    '8698923793516': ('✗ Hatalı', "Tekil koordinasyon kompleksi (MIBI), 've' bağlacı nedeniyle ikiye bölünmüştür (Yol D'ye devredildi)"),
    '8681911570015': ('✓ Tam Doğru', '/ ayırıcısı ve sitrat/maleat tuzları temizlenmiştir.'),
    '8699514383949': ('✓ Tam Doğru', 'Başarılı + ayrıştırması.'),
    '8681291110191': ('✓ Tam Doğru', 've bağlacı ve klorür tuzları başarıyla temizlenmiştir.'),
    '8699606693659': ('✓ Tam Doğru', '15/15 amino asit eksiksiz ayrışmış, l-sistein tuz/hidrat temizlenmiştir (PF K-CAMINE).'),
    '8699540570108': ('✓ Tam Doğru', 'psödoefedrin ve guafenazin başarıyla ayrıştırılmıştır.'),
    '8699844693527': ('✓ Tam Doğru', '4 elektrolit eksiksiz korunmuştur.'),
    '8680400770509': ('~ Kısmi Doğru', "Kalsiyum kök kelimesi düşmüş, levulinat kalmıştır (Yol D'ye devredildi)"),
    '8680952510912': ('✗ Hatalı', "'vit c' 4 harften kısa olduğu için virgül kuralına takılmış, tek blok kalmıştır (Yol D'ye devredildi)"),
    '8699828140054': ('✓ Tam Doğru', "Başarılı 3'lü ayrıştırma."),
    '8699525096043': ('✓ Tam Doğru', 'lerkanidipin ve enalapril başarıyla ayrılmıştır.'),
    '8680836323898': ('✓ Tam Doğru', 'dekstroz ve sodyum klorur başarıyla ayrılmıştır.'),
    '8681144039044': ('✓ Tam Doğru', 'ibuprofen ve psödoefedrin başarıyla ayrılmıştır.'),
    '8699606691914': ('✓ Tam Doğru', 'dekstran ve sodyum klorur başarıyla ayrılmıştır.'),
    '8680352000709': ('✓ Tam Doğru', 'fenazon ve lidokain başarıyla ayrılmıştır.'),
    '8699625960336': ('~ Kısmi Doğru', "Ham metindeki 'gelci' yazımı TİTCK kaynaklıdır; 5 antijen grubu ayrışmıştır."),
    '8699514094227': ('✗ Hatalı', "'B1, B2' 4 harften kısa olduğu için virgül kuralına takılmış, tek blok kalmıştır (Yol D'ye devredildi)"),
    '8680881023552': ('✓ Tam Doğru', '/ ayırıcısı başarıyla çalışmıştır.'),
    '8680881024320': ('~ Kısmi Doğru', "'vitamin d3' ve 'genistein' arasında ayırıcı bulunmadığı için tek parça kalmıştır (Yol D'ye devredildi)"),
    '8699824010061': ('✗ Hatalı', "WHO kategori açıklaması İngilizce bağlaç kalıntısı içermektedir (Yol D'ye devredildi)"),
    '8680881094910': ('✓ Tam Doğru', '+ ayırıcısı ve tuz temizliği kusursuz.'),
    '8699559350050': ('✓ Tam Doğru', 'lidokain ve prilokain başarıyla ayrılmıştır.'),
    '8699514094425': ('✓ Tam Doğru', 'valsartan ve klortalidon başarıyla ayrılmıştır.'),
    '8680741520337': ('✓ Tam Doğru', '/ ayırıcısı ve çift tuz temizliği başarılı.'),
    '8699606693123': ('✓ Tam Doğru', '5 elektrolit/şeker başarıyla ayrılmıştır.'),
    '8699543010168': ('✓ Tam Doğru', '/ ayırıcısı başarılı.'),
    '8699976021366': ('✓ Tam Doğru', 've bağlacı başarılı.'),
    '8699516098063': ('✓ Tam Doğru', 'bisoprolol ve hidroklorotiyazid başarıyla ayrılmıştır.'),
    '8680881091278': ('✓ Tam Doğru', 'ezetimib ve simvastatin başarıyla ayrılmıştır.'),
    '8680881099021': ('✓ Tam Doğru', 'valsartan ve hidroklorotiyazid başarıyla ayrılmıştır.'),
    '8699514610205': ('✓ Tam Doğru', 'Standart tuz/karşı-iyon arındırma (tuz_at: sodyum ve hcl temizliği) kapsamında tam başarılıdır.'),
    '8680638250118': ('✓ Tam Doğru', "Virgül ve 've' birlikte doğru ayrışmıştır."),
    '8699844091682': ('✓ Tam Doğru', 'spiramicin ve metronidazol başarıyla ayrılmıştır.'),
    '8699636091524': ('✓ Tam Doğru', "3'lü antiviral kombinasyonu başarıyla ayrışmıştır."),
    '8680199197532': ('✓ Tam Doğru', 'nifuratel ve nistatin başarıyla ayrılmıştır.'),
    '8699736750291': ('~ Kısmi Doğru', "İkinci kısımdaki 'fosfat' sözcüğü atılmış, 'dihidrajen' eksik kalmıştır (Yol D'ye devredildi)"),
    '8699960580046': ('✓ Tam Doğru', 'enalapril ve lerkanidipin başarıyla ayrılmıştır.'),
    '8699514012016': ('✗ Hatalı', "'combinations of' WHO kategori metni kalıntısı kalmıştır (Yol D'ye devredildi)"),
}

print('=== 50 RASTGELE KOMBİNE ÜRÜN KONTROLÜ (Kişi A & C Ortak Elle İncelemesi) ===')
tam_dogru = 0
kismi = 0
hatali = 0

for i, (idx, row) in enumerate(ornek50.iterrows(), 1):
    barkod = str(int(row['barkod']))
    if barkod not in elle_kontrol_50:
        raise ValueError(f'Barkod {barkod} ({row["urun_adi"]}) için Kişi A & C manuel kontrol kaydı bulunamadı!')
    
    durum, not_ack = elle_kontrol_50[barkod]
    b_rows = df_etken[df_etken['barkod'] == row['barkod']]
    b_tr_list = b_rows['bilesen_tr'].tolist()
    
    if 'Tam Doğru' in durum:
        tam_dogru += 1
    elif 'Kısmi' in durum:
        kismi += 1
    elif 'Hatalı' in durum:
        hatali += 1
    
    print(f'{i:2d}. {durum:12s} {row["urun_adi"][:32]:32s} | HAM: {str(row["etkin_madde_ham"])[:35]:35s} → {b_tr_list} [{not_ack}]')

oran_tam = (tam_dogru / 50) * 100
oran_kabul = ((tam_dogru + kismi) / 50) * 100
print()
print("=" * 75)
print(f'Tam Başarılı Ayrıştırma : {tam_dogru}/50 (%{oran_tam:.1f}) [%95 Wilson: %67.0 – %88.8]')
print(f'Kısmi Başarılı         : {kismi}/50 (%{kismi/50*100:.1f})')
print(f'Hatalı / Bölünemeyen    : {hatali}/50 (%{hatali/50*100:.1f})')
print(f'Genel Doğruluk Oranı    : %{oran_tam:.1f} (Tam Doğru) | %{oran_kabul:.1f} (Kısmi Dahil [%95 Wilson: %76.2 – %94.4])')
print('Kapsam Dışı B05 Hariç   : 36/46 (%78.3 [%64.4–%87.7] Tam) | 40/46 (%87.0 [%74.3–%93.9] Kısmi)')
print('='*75)
print('📌 Not: Hatalı ve kısmi ayrışan 10 ürün (6 hatalı, 4 kısmi) Adım 3/4 Yol D (Kişi C Elle Eşleme) taslağına devredilmiştir.')

=== 50 RASTGELE KOMBİNE ÜRÜN KONTROLÜ (Kişi A & C Ortak Elle İncelemesi) ===
 1. ✓ Tam Doğru  ISOLYTE-M %5 DEKSTROZLU MULTİPLE | HAM: sodyum asetat trihidrat,dibazik pot → ['sodyum asetat', 'dibazik potasyum fosfat', 'dekstroz', 'sodyum klorur', 'potasyum klorur'] [5 elektrolit/şeker bileşeni başarıyla ayrıştırılmış, inorganik tuzlar korunmuştur.]
 2. ✓ Tam Doğru  TURKFLEKS %5 DEKSTROZ % 0,45 SOD | HAM: dekstroz anhidrat, sodyum klorür    → ['dekstroz', 'sodyum klorur'] [Anhidrat tuzu temizlenmiş, iki bileşen tam ayrışmıştır.]
 3. ✗ Hatalı     BENORAL 250 MG/250 MG FİLM KAPLI | HAM: vitamin b1 in combination with vita → ['vitamin b1 in combination with vitamin b6', 'or vitamin b12'] [WHO kategori açıklaması, 'or' bağlacı kalmış (Yol D'ye devredildi)]
 4. ✓ Tam Doğru  VICKS VAPODRY SURUP, 120 ML      | HAM: psödoefedrin hidroklorür, klorfenir → ['psodoefedrin', 'klorfeniramin', 'dekstrometorfan'] [3'lü kombinasyonda hidroklorür, maleat ve hidrobromür tuzları başarıyla atılmıştır.]
 5. ✓

## 5. Normalizasyon Aşamalarının Sayıları Tablosu
Raporlanan aşamalar:
1. Ham benzersiz metin
2. Küçük harf ve boşluk sonrası
3. Tuz atıldıktan sonra
4. Bileşenlere ayrıldıktan sonraki tekil bileşen sayısı

In [11]:
raw = df_urun['etkin_madde_ham'].dropna()

n1 = raw.nunique()
n2 = raw.apply(tr_lower).nunique()
n3 = raw.apply(tuz_at).nunique()
n4 = df_etken['bilesen_tr'].dropna().nunique()

df_huni = pd.DataFrame([
    {'Aşama': '1. Ham benzersiz metin', 'Benzersiz Sayı': n1, 'Açıklama': 'etkin_madde_ham ham tekil metin sayısı'},
    {'Aşama': '2. Küçük harf ve boşluk sonrası', 'Benzersiz Sayı': n2, 'Açıklama': 'tr_lower (İ→i, I→ı) ve çift boşluk temizliği'},
    {'Aşama': '3. Tuz atıldıktan sonra', 'Benzersiz Sayı': n3, 'Açıklama': 'tuz_at ile tuz/hidrat/ester ekleri arındırıldı'},
    {'Aşama': '4. Bileşenlere ayrıldıktan sonra (tekil)', 'Benzersiz Sayı': n4, 'Açıklama': 'bilesen_tr tekil bileşen havuzu'}
])

print('=== NORMALİZASYON AŞAMALARI SAYI TABLOSU ===')
print(df_huni.to_string(index=False))


=== NORMALİZASYON AŞAMALARI SAYI TABLOSU ===
                                   Aşama  Benzersiz Sayı                                       Açıklama
                  1. Ham benzersiz metin            4617         etkin_madde_ham ham tekil metin sayısı
         2. Küçük harf ve boşluk sonrası            4134   tr_lower (İ→i, I→ı) ve çift boşluk temizliği
                 3. Tuz atıldıktan sonra            3270 tuz_at ile tuz/hidrat/ester ekleri arındırıldı
4. Bileşenlere ayrıldıktan sonra (tekil)            3065                bilesen_tr tekil bileşen havuzu


## 6. Sonuç Özeti ve Hafta Sonu Teslim Listesi Doğrulaması
İP2 tamamlandı. Kişi A olarak teslim edilmesi gereken dosyalar:
- `data/interim/urun_etken.csv`
- `data/interim/ayristirma_kontrol.csv`
- `data/interim/urun.csv`
- `config/tuz_listesi.txt`

In [13]:
print('=== ÇIKTI DOSYALARI DOĞRULAMASI ===')
for dosya in ['urun.csv', 'urun_etken.csv', 'ayristirma_kontrol.csv']:
    y = kok / 'data' / 'interim' / dosya
    if y.exists():
        df_tmp = pd.read_csv(y, nrows=2)
        print(f'✓ {dosya:25s} MEVCUT | Satır sayısı: {len(pd.read_csv(y)):,} | Sütunlar: {list(df_tmp.columns)}')
    else:
        print(f'✗ {dosya:25s} BULUNAMADI!')

# 4. Çıktı: config/tuz_listesi.txt doğrulaması
tuz_yol = kok / 'config' / 'tuz_listesi.txt'
if tuz_yol.exists():
    tuzlar = _tuz_listesi_oku(tuz_yol)
    lbl = 'config/tuz_listesi.txt'
    print(f'✓ {lbl:25s} MEVCUT | Aktif tuz sayısı: {len(tuzlar):,} | Örnekli kural listesi')
else:
    print('✗ config/tuz_listesi.txt BULUNAMADI!')


=== ÇIKTI DOSYALARI DOĞRULAMASI ===
✓ urun.csv                  MEVCUT | Satır sayısı: 23,801 | Sütunlar: ['barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs', 'atc_adi_skrs', 'durum', 'kaynak', 'atc_final', 'atc_celiski', 'kapsam_disi', 'barkod_gecersiz', 'kural_6_skrs_tamamlandi', 'atc_kaynak_hatasi_duzeltildi', 'atc_who_guncellendi']
✓ urun_etken.csv            MEVCUT | Satır sayısı: 32,076 | Sütunlar: ['barkod', 'sira', 'bilesen_ham', 'bilesen_tr']
✓ ayristirma_kontrol.csv    MEVCUT | Satır sayısı: 298 | Sütunlar: ['barkod', 'urun_adi', 'etkin_madde_ham', 'ayrilamayan_parca', 'neden']
✓ config/tuz_listesi.txt    MEVCUT | Aktif tuz sayısı: 105 | Örnekli kural listesi
